# QBrain RAG API - End-to-End Demo

This notebook demonstrates the full RAG flow on file:
`D:/Qbrainpython/rag_lab/data/srs/JDECo_SRS.docx[1].pdf`

Flow:
1. Create project
2. Upload SRS and build knowledge source (embeddings)
3. Run retrieval by **project_id** (uses indexed vectors; no re-index)
4. Generate features
5. Read generated features
6. (Optional) Generate test cases for one feature

In [4]:
# Optional install (run once if needed)
# %pip install requests


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import requests

BASE_URL = "http://127.0.0.1:8001"
SRS_FILE = Path(r"D:\\Qbrainpython\\rag_lab\\data\\srs\\JDECo_SRS.docx[1].pdf")
TIMEOUT = 180

assert SRS_FILE.is_file(), f"SRS file not found: {SRS_FILE}"
print("Using file:", SRS_FILE)

def call(method: str, path: str, *, json_body=None, files=None, timeout=TIMEOUT):
    url = f"{BASE_URL}{path}"
    resp = requests.request(method, url, json=json_body, files=files, timeout=timeout)
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f"{method} {path} -> {resp.status_code}")
    return resp.status_code, body


Using file: D:\Qbrainpython\rag_lab\data\srs\JDECo_SRS.docx[1].pdf


In [2]:
# 0) Health check
status, body = call("GET", "/api/v1/health")
print(json.dumps(body, indent=2, ensure_ascii=False))

GET /api/v1/health -> 200
{
  "status": "ok"
}


In [3]:
# 1) Create project
create_payload = {
    "name": "JDECo End-to-End Demo",
    "description": "Notebook API flow demo"
}
status, body = call("POST", "/api/v1/projects/", json_body=create_payload)
print(json.dumps(body, indent=2, ensure_ascii=False))

assert status == 200, body
project_id = body["id"]
print("project_id:", project_id)

POST /api/v1/projects/ -> 200
{
  "id": "bb496ed8-7166-4e44-939a-c8f1020259fc",
  "name": "JDECo End-to-End Demo",
  "doc_path": null,
  "description": "Notebook API flow demo"
}
project_id: bb496ed8-7166-4e44-939a-c8f1020259fc


In [4]:
# 2) Upload SRS and build knowledge source
with SRS_FILE.open("rb") as f:
    files = {"srs": (SRS_FILE.name, f, "application/pdf")}
    status, body = call("POST", f"/api/v1/projects/{project_id}/upload-srs", files=files, timeout=600)

print(json.dumps(body, indent=2, ensure_ascii=False))
assert status == 200, body

processed_doc_path = body.get("doc_path")
print("processed_doc_path:", processed_doc_path)
print("processing summary:", body.get("processing"))

POST /api/v1/projects/bb496ed8-7166-4e44-939a-c8f1020259fc/upload-srs -> 200
{
  "projectId": "bb496ed8-7166-4e44-939a-c8f1020259fc",
  "filename": "JDECo_SRS.docx[1].pdf",
  "stored_as": "bb496ed8-7166-4e44-939a-c8f1020259fc_7889f5749f3948a6bcfb5c46114eb545.pdf",
  "doc_path": "D:\\Qbrainpython\\rag_lab\\data\\srs\\bb496ed8-7166-4e44-939a-c8f1020259fc_7889f5749f3948a6bcfb5c46114eb545.pdf",
  "processing": {
    "chunk_count": 30,
    "embeddings_indexed": 30,
    "knowledge_source_ready": true,
    "feature_count": 0,
    "test_case_count": 0
  }
}
processed_doc_path: D:\Qbrainpython\rag_lab\data\srs\bb496ed8-7166-4e44-939a-c8f1020259fc_7889f5749f3948a6bcfb5c46114eb545.pdf
processing summary: {'chunk_count': 30, 'embeddings_indexed': 30, 'knowledge_source_ready': True, 'feature_count': 0, 'test_case_count': 0}


In [5]:
# 3) Retrieval endpoint by project_id (top-k chunks)
# This uses vectors already indexed by upload-srs.
retrieval_payload = {
    "query": "What should the system display while a function is being processed?",
    "k": 5
}
status, body = call("POST", f"/api/v1/projects/{project_id}/retrieval", json_body=retrieval_payload)
assert status == 200, body

print("results count:", len(body.get("results", [])))
for i, hit in enumerate(body.get("results", []), 1):
    print(f"\\n--- Hit {i} ---")
    print("metadata:", hit.get("metadata", {}))
    print((hit.get("page_content", "") or "")[:500])

POST /api/v1/projects/bb496ed8-7166-4e44-939a-c8f1020259fc/retrieval -> 200
results count: 5
\n--- Hit 1 ---
metadata: {'chunk_id': 23, 'project_id': 'bb496ed8-7166-4e44-939a-c8f1020259fc', 'source_file': 'bb496ed8-7166-4e44-939a-c8f1020259fc_7889f5749f3948a6bcfb5c46114eb545.pdf'}
Software Requirements Specification for JDECo Services Management system​
Page 28 
 
UL-5 
when a function is being processed by system,the system shall 
display an icon indicating the processing . 
UL-6 
The system shall provide a help link from each displayed page to 
explain how to use this page. 
UL-7 
system shall allow users to navigate  customers  ,using mouse in web 
application and touch screen in mobile app . 
UL-8 
all icons of navigation must change color when the mouse is over 
them
\n--- Hit 2 ---
metadata: {'chunk_id': 22, 'project_id': 'bb496ed8-7166-4e44-939a-c8f1020259fc', 'source_file': 'bb496ed8-7166-4e44-939a-c8f1020259fc_7889f5749f3948a6bcfb5c46114eb545.pdf'}
Software Requirements Specif

In [6]:
# 4) Generate features from processed doc
# skip_tests=True means generate features only (faster)
feature_payload = {
    "skip_tests": True,
    "test_context_k": 5
}
status, body = call("POST", f"/api/v1/projects/{project_id}/extract-features-from-doc", json_body=feature_payload, timeout=600)
assert status == 200, body

print("feature_count (pipeline metadata):", body.get("metadata", {}).get("feature_count"))
print("persisted:", body.get("persisted"))
print("sample feature names:")
for x in (body.get("features") or [])[:10]:
    print("-", x.get("name"))

POST /api/v1/projects/bb496ed8-7166-4e44-939a-c8f1020259fc/extract-features-from-doc -> 200
feature_count (pipeline metadata): 57
persisted: {'feature_count': 57, 'test_case_count': 0}
sample feature names:
- Service Request Submission
- Service Request Update
- Service Request Cancellation
- Service Invoice Payment
- Create/Update Service
- Create Payment Requests
- Create Invoice
- Manage Inventory
- Install Infrastructure
- Recover Incomplete Service Requests


In [7]:
# 5) Read saved features from API table endpoint
status, saved_features = call("GET", f"/api/v1/features/projects/{project_id}/features")
assert status == 200, saved_features
print("saved features:", len(saved_features))
for f in saved_features[:10]:
    print("-", f.get("id"), "|", f.get("title"))

GET /api/v1/features/projects/bb496ed8-7166-4e44-939a-c8f1020259fc/features -> 200
saved features: 96
- 38ca3b0d-4fe6-425a-a3dc-fcdc8190dee5 | Customer Service Viewing Restrictions
- e44a32fb-ffd6-4658-aeff-7943797b8477 | Administrator Account Creation
- d3bce1ad-27ba-42fb-a1ee-924c80472410 | User Login Requirement
- 88e64c00-cdc6-48d8-b515-eef4e218e357 | Encryption of Sensitive Transactions
- ed5f7d39-e66f-4643-87c1-fef0ba2650c3 | Peak User Accommodation
- aed09deb-505c-464b-8110-75f58265d108 | Usability Training Requirement
- e1bb6422-2151-47a1-bbb4-23b3d71b6a91 | Processing Indicator
- f9e3fb3a-066e-4d67-bebd-cf440802cd43 | Process Confirmation Message
- 24a0b10c-5aba-4cc0-8902-1b50d75cb5b7 | Home Icon Navigation
- d1046b66-a2f8-4a75-ba06-bc067d7e90f3 | Retention of Services List


In [8]:
# 6) Optional: generate test cases for first feature
# Set RUN_TEST_CASE_GEN=True if you want to run it.
RUN_TEST_CASE_GEN = False

if RUN_TEST_CASE_GEN and saved_features:
    first_feature_id = saved_features[0]["id"]
    tc_payload = {"test_context_k": 5}
    status, tc_body = call(
        "POST",
        f"/api/v1/test-cases/features/{first_feature_id}/generate-test-cases",
        json_body=tc_payload,
        timeout=600,
    )
    print("status:", status)
    print(json.dumps(tc_body, indent=2, ensure_ascii=False)[:4000])
else:
    print("Skipped test-case generation. Set RUN_TEST_CASE_GEN=True to enable.")

Skipped test-case generation. Set RUN_TEST_CASE_GEN=True to enable.
